In [2]:

import os
import unstructured_pytesseract

tesseract_path = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Verify that the executable exists
assert os.path.isfile(tesseract_path), (
    f"Tesseract not found at: {tesseract_path}"
)

# Tell Unstructured's OCR wrapper where Tesseract is
unstructured_pytesseract.pytesseract.tesseract_cmd = (
    tesseract_path
)

from unstructured.partition.pdf import partition_pdf

# Keep your existing partition_pdf() code below


from unstructured.partition.pdf import partition_pdf



c:\Users\debra\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
output_path = "./content/"
file_path = output_path + 'DM2.pdf'

# Reference: https://docs.unstructured.io/open-source/core-functionality/chunking
chunks = partition_pdf(
    filename=file_path,
    infer_table_structure=True,            # extract tables
    strategy="hi_res",                     # mandatory to infer tables

    extract_image_block_types=["Image", "Formula"],   # Add 'Table' to list to extract image of tables
    image_output_dir_path=output_path,   # if None, images and tables will saved in base64

    extract_image_block_to_payload=True,   # if true, will extract base64 for API usage

    chunking_strategy="by_title",          # or 'basic'
    max_characters=10000,                  # defaults to 500
    combine_text_under_n_chars=2000,       # defaults to 0
    new_after_n_chars=6000,

    # extract_images_in_pdf=True,          # deprecated
)

Loading weights: 100%|██████████| 367/367 [00:00<00:00, 1581.68it/s]


In [ ]:
# We get 2 types of elements from the partition_pdf function
#set([str(type(el)) for el in chunks])
#print(len(chunks))

{"<class 'unstructured.documents.elements.CompositeElement'>"}

In [43]:
# Each CompositeElement containes a bunch of related elements.
# This makes it easy to use these elements together in a RAG pipeline.

chunks[6].metadata.orig_elements


In [ ]:
# This is what an extracted image looks like.
# It contains the base64 representation only because we set the param extract_image_block_to_payload=True

#elements = chunks[0].metadata.orig_elements
#chunk_images = [el for el in elements if 'Text' in str(type(el))]
#chunk_images[1].to_dict()

{'type': 'NarrativeText',
 'element_id': '27d67f4d-9631-40a1-acd0-4346fb5fa193',
 'text': 'Carla Biggio ,1 Lorenzo Calibbi ,2 Toshihiko Ota ,3,4 and Samuele Zanchini 1',
 'metadata': {'detection_class_prob': 0.7715966701507568,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(368.3463888888889),
     np.float64(263.3449305555555)),
    (np.float64(368.3463888888889), np.float64(298.7997827777778)),
    (np.float64(1332.2020263671875), np.float64(298.7997827777778)),
    (np.float64(1332.2020263671875), np.float64(263.3449305555555))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'links': [{'text': ',',
    'url': 'https://orcid.org/0000-0001-9688-2439',
    'start_index': 13},
   {'text': ',',
    'url': 'https://orcid.org/0000-0001-8193-5665',
    'start_index': 17}],
  'last_modified': '2026-10-19T00:17:35',
  'filetype': 'PNG',
  'languages': ['eng'],
  'page_number': 1}}

In [7]:
# separate tables from texts
tables = []
texts = []

for chunk in chunks:
    if "Table" in str(type(chunk)):
        tables.append(chunk)

    if "CompositeElement" in str(type((chunk))):
        texts.append(chunk)

In [8]:
# Get the images from the CompositeElement objects
def get_images_base64(chunks):
    images_b64 = []
    image_captions = []
    for chunk in chunks:
        if "CompositeElement" in str(type(chunk)):
            chunk_els = chunk.metadata.orig_elements
            for i, el in enumerate(chunk_els):
                if "Image" in str(type(el)):
                    images_b64.append(el.metadata.image_base64)
                    
                    # Prevent another potential IndexError if an Image is the absolute last element
                    if i + 1 < len(chunk_els) and "FigureCaption" in str(type(chunk_els[i+1])):
                        image_captions.append(chunk_els[i+1].text)
                        
                        # Use -2 to check the previously added caption
                        if len(image_captions) > 1 and image_captions[-2] == "None":
                            image_captions[-2] = chunk_els[i+1].text
                    else:
                        image_captions.append("None")
    return images_b64, image_captions


# Get the formula from the CompositeElement objects
def get_formulas_base64(chunks):
    formulas_b64 = []
    for chunk in chunks:
        if "CompositeElement" in str(type(chunk)):
            chunk_els = chunk.metadata.orig_elements
            for el in chunk_els:
                if "Formula" in str(type(el)):
                    formulas_b64.append(el.metadata.image_base64)
    return formulas_b64

images, captions = get_images_base64(chunks)
formulas = get_formulas_base64(chunks)

In [ ]:
import base64
'''
from IPython.display import Image, display

def display_base64_image(base64_code):
    # Decode the base64 string to binary
    image_data = base64.b64decode(base64_code)
    # Display the image
    display(Image(data=image_data))

display_base64_image(images[2])
'''

'\nimport base64\nfrom IPython.display import Image, display\n\ndef display_base64_image(base64_code):\n    # Decode the base64 string to binary\n    image_data = base64.b64decode(base64_code)\n    # Display the image\n    display(Image(data=image_data))\n\ndisplay_base64_image(images[2])\n'

In [30]:
#print(captions[2])

In [11]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage

In [12]:
#Prompt
prompt_text = """
You are an assistant tasked with summarizing tables and text
from scientific research papers.

Give a concise, accurate summary of the supplied table or text.
Preserve technical terminology, numerical values, and key findings, mention authors of the paper when required.

Respond only with the summary.

Table or text chunk:
{element}
"""

prompt = ChatPromptTemplate.from_template(prompt_text)


#Summary chain
model = ChatOllama(
    model="qwen2.5vl:3b",
    temperature=0.5
)

summarize_chain = (
    {"element": lambda x: x}
    | prompt
    | model
    | StrOutputParser()
)

In [13]:
text_summaries = summarize_chain.batch(
    texts,
    config={"max_concurrency": 3}
)

tables_html = [
    table.metadata.text_as_html
    for table in tables
]

table_summaries = summarize_chain.batch(
    tables_html,
    config={"max_concurrency": 2}
)

In [31]:
#text_summaries

In [32]:
#print(len(images))
#print(len(captions))
#captions[0]

In [16]:
prompt_template = """
You are analyzing a figure extracted from a scientific research paper.

Your task is to produce a scientifically accurate description of the image, using the provided figure caption as essential context.

FIGURE CAPTION:
{caption}

INSTRUCTIONS:

1. Read and interpret the figure caption BEFORE analyzing the image. The caption describes the scientific purpose, subject, and context of the figure. Use this information to guide your interpretation of the visual content.

2. Identify the type of figure: plot, diagram, schematic, photograph, equation, table, or a combination of these.

3. Connect the visible content to the caption:
   - Identify the objects, physical quantities, processes, or theoretical concepts mentioned in the caption that are actually visible in the image.
   - Explain how the visible elements relate to the scientific subject described in the caption.
   - Use terminology from the caption where appropriate.
   - If the caption specifies a physical process, model, experimental setup, or theoretical relationship, interpret the image in that context.

4. If the image contains graphs, describe the axes, variables, units, curves, legends, trends, important features, and relationships. Use the caption to interpret what the plotted quantities represent. Do not invent axis labels, numerical values, or trends that cannot be established from the image.

5. If the image contains a schematic or scientific diagram, describe its components, labels, arrows, connections, and the physical or mathematical process they represent, informed by the caption.

6. If the image contains multiple panels, describe each panel separately and explain how the panels relate to the caption and to each other.

7. Distinguish between information explicitly provided by the caption and information directly observable in the image. Do not invent scientific conclusions that are unsupported by either source.

8. If the caption describes something that is not clearly visible, acknowledge the uncertainty rather than ignoring the caption or hallucinating visual details.

OUTPUT REQUIREMENTS:
Write a detailed, coherent scientific description suitable for indexing and retrieval in a retrieval-augmented generation (RAG) system. Prioritize scientific meaning and relationships over superficial visual appearance.

Return only the image summary. Do not repeat these instructions.
"""

def summarize_image(image, caption):
    # Strip the data URL prefix if present
    if image.startswith("data:"):
        header, image = image.split(",", 1)
        mime_type = header.split(";")[0].split(":")[1]
    else:
        mime_type = "image/jpeg"

    # Validate that the image is base64
    base64.b64decode(image, validate=True)

    message = HumanMessage(
        content=[
            {
                "type": "text",
                "text": prompt_template.format(
                    caption=caption or ""
                )
            },
            {
                "type": "image",
                "base64": image,
                "mime_type": mime_type
            }
        ]
    )

    response = model.invoke([message])
    return StrOutputParser().invoke(response)


assert len(images) == len(captions)

image_summaries = [
    summarize_image(image, caption)
    for image, caption in zip(images, captions)
]

In [33]:
#image_summaries

In [34]:
#print(len(image_summaries))

In [19]:
import uuid
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import MultiVectorRetriever

In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "normalize_embeddings": True
    }
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19620.19it/s]


In [21]:
# The vectorstore to use to index the child chunks
vectorstore = Chroma(collection_name="multi_modal_rag", embedding_function=embeddings)

# The storage layer for the parent documents
store = InMemoryStore()
id_key = "doc_id"

# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

In [36]:
# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [
    Document(page_content=summary, metadata={id_key: doc_ids[i]}) for i, summary in enumerate(text_summaries)
]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

# Add tables
table_ids = [str(uuid.uuid4()) for _ in tables]
summary_tables = [
    Document(page_content=summary, metadata={id_key: table_ids[i]}) for i, summary in enumerate(table_summaries)
]
if summary_tables:
    retriever.vectorstore.add_documents(summary_tables)
    retriever.docstore.mset(list(zip(table_ids, tables)))

# Add image summaries
img_ids = [str(uuid.uuid4()) for _ in images]
summary_img = [
    Document(page_content=summary, metadata={id_key: img_ids[i]}) for i, summary in enumerate(image_summaries)
]
if summary_img:
    retriever.vectorstore.add_documents(summary_img)
    retriever.docstore.mset(list(zip(img_ids, images)))

In [37]:
# Retrieve
docs = retriever.invoke(
    "who are the authors of this paper?"
)

In [38]:
for doc in docs:
    print(str(doc) + "\n\n" + "-" * 80)

PHYS. REV. D 108, 115003 (2023)

[144] F. Takahashi, M. Yamada, and W. Yin, What if ALP dark matter for the XENON1T excess is the inflaton, J. High Energy Phys. 01 (2021) 152.

[145] D. Buttazzo, P. Panci, D. Teresi, and R. Ziegler, Xenon1T excess from electron recoils of non-relativistic dark matter, Phys. Lett. B 817, 136310 (2021).

[146] C. Han, M. L. López-Ibáñez, A. Melis, O. Vives, and J. M. Yang, Anomaly-free ALP from non-Abelian flavor sym- metry, J. High Energy Phys. 08 (2022) 306.

[147] K. Sakurai and F. Takahashi, Anomaly-free axion dark matter in three Higgs doublet model and its pheno- menological implications, J. High Energy Phys. 07 (2022) 124.

[148] G. G. Raffelt, Astrophysical methods to constrain axions and other novel particle phenomena, Phys. Rep. 198, 1 (1990).

[149] G. Raffelt and A. Weiss, Red giant bound on the axion— electron coupling revisited, Phys. Rev. D 51, 1495 (1995).

[150] N. Viaux, M. Catelan, P. B. Stetson, G. Raffelt, J. Redondo, A. A. R. Valcar

In [39]:
for i, doc in enumerate(docs):
    if "CompositeElement" in str(type(doc)):
        print("\n\nChunk", i)
        for test in doc.metadata.orig_elements:
            print(test.to_dict()["type"], test.metadata.page_number)



Chunk 0
UncategorizedText 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
NarrativeText 26
Header 27
UncategorizedText 27
ListItem 27
ListItem 27
ListItem 27
UncategorizedText 27
ListItem 27
ListItem 27
ListItem 27
ListItem 27
ListItem 27
UncategorizedText 27


Chunk 1
UncategorizedText 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
NarrativeText 26
Header 27
UncategorizedText 27
ListItem 27
ListItem 27
ListItem 27
UncategorizedText 27
ListItem 27
ListItem 27
ListItem 27
ListItem 27
ListItem 27
UncategorizedText 27


Chunk 2
UncategorizedText 26
ListItem 26
ListItem 26
ListItem 26
ListItem 26
ListIte

In [26]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage, HumanMessage
#from langchain_ollama import ChatOllama
from base64 import b64decode


def parse_docs(docs):
    """Split base64-encoded images and texts"""
    b64 = []
    text = []
    for doc in docs:
        try:
            b64decode(doc)
            b64.append(doc)
        except Exception as e:
            text.append(doc)
    return {"images": b64, "texts": text}


def build_prompt(kwargs):

    docs_by_type = kwargs["context"]
    user_question = kwargs["question"]

    context_text = ""
    if len(docs_by_type["texts"]) > 0:
        for text_element in docs_by_type["texts"]:
            context_text += text_element.text

    # construct prompt with context (including images)
    prompt_template = f"""
    Answer the question based only on the following context, which can include text, tables, and the below image.
    Context: {context_text}
    Question: {user_question}
    """

    prompt_content = [{"type": "text", "text": prompt_template}]

    if len(docs_by_type["images"]) > 0:
        for image in docs_by_type["images"]:
            prompt_content.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image}"},
                }
            )

    return ChatPromptTemplate.from_messages(
        [
            HumanMessage(content=prompt_content),
        ]
    )


chain = (
    {
        "context": retriever | RunnableLambda(parse_docs),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(build_prompt)
    | model
    | StrOutputParser()
)

chain_with_sources = {
    "context": retriever | RunnableLambda(parse_docs),
    "question": RunnablePassthrough(),
} | RunnablePassthrough().assign(
    response=(
        RunnableLambda(build_prompt)
        | model
        | StrOutputParser()
    )
)

In [40]:
response = chain.invoke(
    "what is the main result of this paper?"
)

print(response)

ResponseError: request (6743 tokens) exceeds the available context size (4096 tokens), try increasing it (status code: 400)